Azure offers a broad portfolio of managed database services. This topic covers the two most widely used:

- **Azure SQL** — the family of managed relational database services built on SQL Server
- **Azure Cosmos DB** — a globally distributed, multi-model NoSQL database with single-digit millisecond latency

| Service | AWS equivalent | Type |
|---|---|---|
| Azure SQL Database | Amazon RDS (SQL Server) | Fully managed relational PaaS |
| Azure SQL Managed Instance | Amazon RDS Custom | Near-100% SQL Server compatibility, VNet-injected |
| SQL Server on Azure VM | EC2 + SQL Server | IaaS — full control |
| Azure Cosmos DB | Amazon DynamoDB + DocumentDB | Multi-model NoSQL, globally distributed |

## Azure SQL Family

### Azure SQL Database

**Azure SQL Database** is a fully managed, serverless-capable PaaS database built on the latest stable SQL Server engine. Azure handles OS patching, backups, high availability, and scaling.

#### Deployment models

| Model | Description |
|---|---|
| **Single database** | One database with its own guaranteed compute resources |
| **Elastic pool** | Multiple databases sharing a pool of compute resources — cost-effective for many databases with variable load |
| **Serverless** | Auto-pauses when idle (billing stops); auto-resumes on connection; min/max vCore range |

#### Service tiers (purchasing models)

Azure SQL Database uses two purchasing models:

**vCore model** (recommended):

| Tier | Description | Use case |
|---|---|---|
| **General Purpose** | Balanced compute and storage (remote SSD) | Most workloads |
| **Business Critical** | Local SSD; 3-node AG cluster; readable secondary | High I/O, low latency; read scale-out |
| **Hyperscale** | Up to 100 TiB; distributed storage; scale-out read replicas | Very large databases |

**DTU model** (legacy): bundled compute+I/O+storage units — Basic, Standard, Premium tiers.

#### High availability

- **General Purpose** — relies on Azure Storage redundancy; failover ~20–30 seconds
- **Business Critical** — built-in Always On Availability Group across 3 replicas; one readable secondary included; failover ~5–10 seconds
- **Hyperscale** — compute nodes are stateless; read replicas can be added and scale instantly

#### Backups

Azure SQL Database takes backups automatically:
- **Full backup** — weekly
- **Differential backup** — every 12–24 hours
- **Log backup** — every 5–10 minutes

Retention: 1–35 days (configurable). **Long-Term Retention (LTR)** extends retention to up to 10 years using Azure Blob Storage.

**Point-in-time restore (PITR)** lets you restore to any point within the retention window.

#### Serverless tier

The **Serverless** compute tier auto-scales vCores between a configured min and max based on demand, and **auto-pauses** the database after a configurable idle period (minimum 1 hour). Billing stops during the pause. On the next connection, the database auto-resumes in ~1 minute. Ideal for development databases and workloads with intermittent, unpredictable usage.

### Azure SQL Managed Instance

**Azure SQL Managed Instance (SQL MI)** provides near-100% compatibility with on-premises SQL Server Enterprise Edition — including SQL Agent, cross-database queries, linked servers, CLR, Service Broker, and Database Mail. It is deployed inside a **dedicated VNet subnet** (VNet injection), giving it a private IP and making it invisible to the internet.

| Feature | SQL Database | SQL Managed Instance |
|---|---|---|
| SQL Server compatibility | ~95% | ~100% |
| VNet deployment | No (private endpoint) | Yes (VNet injection) |
| SQL Agent | No | Yes |
| Cross-database queries | No | Yes |
| CLR, Service Broker | No | Yes |
| Linked servers | No | Yes |
| Instance-level collation | No | Yes |
| Migration from on-prem | Requires refactoring | Lift-and-shift |

SQL MI is the right choice when migrating legacy on-premises SQL Server workloads that depend on instance-level features that SQL Database does not support.

## Azure SQL Security and Networking

### Authentication

| Method | Notes |
|---|---|
| **Entra ID authentication** | Users and service principals authenticate with Entra ID token — recommended |
| **SQL authentication** | Username + password in the database — avoid for production |
| **Entra ID + SQL** | Both enabled simultaneously |

Set an **Entra ID admin** on the SQL server — this account can then grant database access to other Entra ID users and groups using standard `CREATE USER ... FROM EXTERNAL PROVIDER` syntax.

### Network access

| Option | Description |
|---|---|
| **Public endpoint + firewall rules** | Allow specific IP ranges or Azure services; default |
| **Service endpoint** | Route VNet traffic to SQL over Azure backbone; SQL still has public IP |
| **Private endpoint** | Private IP in VNet (`privatelink.database.windows.net`); recommended for production |

### Transparent Data Encryption (TDE)

All data at rest is encrypted by default using **TDE** with AES-256. You can use:
- **Service-managed key** — Azure manages the key (default)
- **Customer-managed key (BYOK)** — key stored in Azure Key Vault; you control rotation and can revoke access

### Advanced security features

| Feature | Description |
|---|---|
| **Microsoft Defender for SQL** | Vulnerability assessment + Advanced Threat Protection (detects SQL injection, unusual access patterns) |
| **Auditing** | Log all database events to Log Analytics, Storage, or Event Hub |
| **Dynamic Data Masking** | Mask sensitive columns (e.g. credit card numbers) for non-privileged users — without changing stored data |
| **Always Encrypted** | Column-level encryption; key never exposed to Azure — only the client application holds the key |
| **Row-Level Security** | Filter rows returned to a user based on their identity |

## Azure Cosmos DB

**Azure Cosmos DB** is a fully managed, globally distributed, multi-model NoSQL database. It guarantees **single-digit millisecond latency** at the 99th percentile for both reads and writes, **99.999% availability SLA** with multi-region writes, and **automatic indexing** of all fields.

The AWS equivalent is a combination of DynamoDB (key-value/document) + DocumentDB (MongoDB API).

### Resource hierarchy

```
Cosmos DB Account
 └── Database
       └── Container  (the unit of scale — throughput and storage are per-container)
             └── Items  (documents, rows, nodes, edges — depending on the API)
```

### APIs

Cosmos DB is multi-model — you choose the API when creating the account:

| API | Data model | AWS equivalent | Use case |
|---|---|---|---|
| **NoSQL (Core SQL)** | JSON documents, SQL-like query | DynamoDB | Default; richest Cosmos DB feature set |
| **MongoDB** | BSON documents, MongoDB wire protocol | DocumentDB | Migrate existing MongoDB apps |
| **Cassandra** | Wide-column, CQL | Keyspaces | Migrate Cassandra workloads |
| **Gremlin** | Graph (vertices + edges) | Neptune | Social graphs, recommendation engines |
| **Table** | Key-value (PartitionKey + RowKey) | DynamoDB | Migrate Azure Table Storage workloads |
| **PostgreSQL (vCore)** | Relational — distributed Citus | Aurora | Sharded PostgreSQL at scale |

### Partition key

The **partition key** is the most important design decision in Cosmos DB. It determines how data is physically distributed across partitions:

- Choose a key with **high cardinality** — many distinct values
- Choose a key that **spreads reads and writes evenly** — avoid hot partitions
- Queries that include the partition key are **efficient** (single-partition); cross-partition queries fan out and are more expensive
- A logical partition can hold up to **20 GB**
- Synthetic partition keys: combine two fields (e.g. `tenantId + date`) for better distribution

## Cosmos DB Throughput and Capacity

### Request Units (RUs)

Cosmos DB measures throughput in **Request Units per second (RU/s)**. One RU is the cost of reading a 1 KB document by its ID. All operations — reads, writes, queries, stored procedures — consume RUs.

| Operation | Approximate RU cost |
|---|---|
| Point read (1 KB by ID) | 1 RU |
| Write (1 KB) | ~5 RUs |
| Query (simple, single partition) | ~2.5 RUs |
| Cross-partition query | Higher — proportional to partitions scanned |

### Throughput provisioning modes

| Mode | Description | Use case |
|---|---|---|
| **Manual provisioned** | Fixed RU/s; you pay regardless of usage | Predictable, steady workloads |
| **Autoscale** | Scales between 10% and 100% of max RU/s automatically; you pay for peak reached | Variable or unpredictable workloads |
| **Serverless** | Pay per RU consumed; no provisioned throughput | Dev/test; sporadic workloads |

Throughput can be provisioned at the **database level** (shared across all containers) or at the **container level** (dedicated to that container).

### Free tier

Cosmos DB has a **free tier**: 1,000 RU/s and 25 GB of storage per account, perpetually free — one free tier account per subscription.

## Cosmos DB Global Distribution

You can add or remove Azure regions to a Cosmos DB account with a single click. Data is automatically replicated to all configured regions.

### Consistency levels

Cosmos DB offers five tunable consistency levels — a sliding scale between strong consistency and high availability/performance:

| Level | Guarantee | Latency | Availability | Use case |
|---|---|---|---|---|
| **Strong** | Read always returns the latest write (linearisable) | Highest | Lower | Financial transactions |
| **Bounded Staleness** | Reads lag behind writes by at most K versions or T seconds | High | Lower | Near-real-time leaderboards |
| **Session** (default) | Consistent within a client session; stale for other clients | Medium | High | User profiles, shopping carts |
| **Consistent Prefix** | Reads never see out-of-order writes | Low | High | Social feeds, IoT telemetry |
| **Eventual** | No ordering guarantee; lowest latency | Lowest | Highest | Click counting, likes |

> **Session consistency** is the default and works well for most user-facing applications — a user always reads their own writes.

### Multi-region writes

By default, one region is the **write region** and others are **read regions**. Enabling **multi-region writes** (active-active) allows writes in any region — providing the lowest write latency globally. Conflict resolution is handled by:

| Policy | Description |
|---|---|
| **Last-Writer-Wins (LWW)** | Default — conflict resolved by a timestamp property |
| **Custom (merge procedure)** | JavaScript stored procedure called on conflict |

## Cosmos DB Key Features

### Indexing

Cosmos DB **automatically indexes every property** in every document by default — no schema definition needed. You can customise the indexing policy per container:

- **Exclude paths** — reduce RU cost and storage for fields never queried
- **Composite indexes** — required for ORDER BY on multiple fields
- **Spatial indexes** — for geospatial queries

### Change Feed

The **Change Feed** is a persistent, ordered log of every insert and update to items in a container. Deletes are not captured by default (a soft-delete pattern is recommended).

Use cases:
- Trigger Azure Functions on data changes (event-driven architecture)
- Materialize read-optimised views in another container
- Real-time stream processing with Azure Stream Analytics
- Audit log

The Change Feed is read using the **Change Feed processor** (SDK) or the Cosmos DB trigger in Azure Functions.

### Time to Live (TTL)

Set a **TTL** on a container or per-item to automatically expire and delete items after a specified number of seconds. Useful for session data, temporary tokens, and log records with a retention policy.

### Analytical store

Enable the **Analytical Store** (Azure Synapse Link) to get a fully isolated, auto-synced columnar copy of your Cosmos DB data for analytics — without impacting transactional workloads. Query with Azure Synapse Analytics (Spark or SQL Serverless) without ETL.

## Choosing Between Azure SQL and Cosmos DB

| Scenario | Use |
|---|---|
| Relational data with complex joins and transactions | **Azure SQL Database** |
| Lift-and-shift SQL Server with instance-level features | **Azure SQL Managed Instance** |
| Full SQL Server control (CLR, custom OS config) | **SQL Server on Azure VM** |
| Global low-latency reads and writes (<10 ms) | **Cosmos DB** |
| Document store (JSON), flexible schema | **Cosmos DB NoSQL API** |
| Existing MongoDB application | **Cosmos DB MongoDB API** |
| Very large database (>4 TiB) | **Azure SQL Hyperscale** or **Cosmos DB** |
| Event-driven architecture with change feed | **Cosmos DB** |
| Dev/test, intermittent usage, cost sensitive | **Azure SQL Serverless** or **Cosmos DB Serverless** |

## Working with Azure SQL and Cosmos DB via Python

In [ ]:
# pip install azure-identity azure-mgmt-sql azure-cosmos pyodbc

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.mgmt.sql import SqlManagementClient
from azure.mgmt.sql.models import Server, Database, FirewallRule, BackupShortTermRetentionPolicy

credential      = DefaultAzureCredential()
subscription_id = "<your-subscription-id>"
resource_group  = "my-rg"
location        = "eastus"

sql = SqlManagementClient(credential, subscription_id)

In [ ]:
# Create a logical SQL server
poller = sql.servers.begin_create_or_update(
    resource_group, "my-sql-server",
    Server(
        location=location,
        administrator_login="sqladmin",
        administrator_login_password="<strong-password>",
        version="12.0",
        minimal_tls_version="1.2"
    )
)
server = poller.result()
print(f"Server: {server.name}  FQDN: {server.fully_qualified_domain_name}")

In [ ]:
# Create a General Purpose vCore database (serverless)
poller = sql.databases.begin_create_or_update(
    resource_group, "my-sql-server", "my-database",
    Database(
        location=location,
        sku={"name": "GP_S_Gen5", "tier": "GeneralPurpose", "family": "Gen5", "capacity": 2},
        auto_pause_delay=60,          # pause after 60 min idle
        min_capacity=0.5,             # min 0.5 vCores when active
    )
)
db = poller.result()
print(f"Database: {db.name}  tier: {db.sku.tier}  status: {db.status}")

# Set short-term backup retention to 14 days
sql.backup_short_term_retention_policies.begin_create_or_update(
    resource_group, "my-sql-server", "my-database", "default",
    BackupShortTermRetentionPolicy(retention_days=14)
).result()
print("Backup retention set to 14 days")

In [ ]:
# Allow Azure services to access the SQL server
sql.firewall_rules.create_or_update(
    resource_group, "my-sql-server", "AllowAzureServices",
    FirewallRule(start_ip_address="0.0.0.0", end_ip_address="0.0.0.0")
)

# List databases on the server
for db in sql.databases.list_by_server(resource_group, "my-sql-server"):
    print(f"  {db.name:<30} {db.sku.tier if db.sku else 'system':<20} status: {db.status}")

In [ ]:
import struct, pyodbc

# Connect to Azure SQL using Entra ID token (Managed Identity or interactive)
token = credential.get_token("https://database.windows.net/.default").token
token_bytes = token.encode("utf-16-le")
token_struct = struct.pack(f"<I{len(token_bytes)}s", len(token_bytes), token_bytes)

conn_str = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=my-sql-server.database.windows.net;"
    "DATABASE=my-database;"
    "Encrypt=yes;"
)
conn = pyodbc.connect(conn_str, attrs_before={1256: token_struct})
cursor = conn.cursor()
cursor.execute("SELECT @@VERSION")
print(cursor.fetchone()[0])

In [ ]:
from azure.cosmos import CosmosClient, PartitionKey, exceptions

# Connect using Entra ID (DefaultAzureCredential)
cosmos_url = "https://my-cosmos-account.documents.azure.com:443/"
cosmos_client = CosmosClient(url=cosmos_url, credential=credential)

# Create database and container
db_client = cosmos_client.create_database_if_not_exists(id="my-db")

container_client = db_client.create_container_if_not_exists(
    id="orders",
    partition_key=PartitionKey(path="/customerId"),
    offer_throughput=400,              # 400 RU/s manual
    default_ttl=86400 * 30             # 30-day TTL on all items
)
print(f"Container: {container_client.id}")

In [ ]:
import uuid

# Create items
order = {
    "id": str(uuid.uuid4()),
    "customerId": "cust-001",
    "product": "Azure T-shirt",
    "quantity": 2,
    "status": "pending"
}
response = container_client.create_item(body=order)
print(f"Created item: {response['id']}  RU charge: {response['_etag']}")

# Point read (cheapest — uses partition key + item id)
item = container_client.read_item(item=order["id"], partition_key="cust-001")
print(f"Read: {item['product']}")

# Query within a partition
query = "SELECT * FROM orders o WHERE o.customerId = @cid AND o.status = @status"
params = [{"name": "@cid", "value": "cust-001"}, {"name": "@status", "value": "pending"}]
for item in container_client.query_items(query=query, parameters=params,
                                         partition_key="cust-001"):
    print(f"  Order: {item['id']}  product: {item['product']}")

# Upsert
order["status"] = "shipped"
container_client.upsert_item(body=order)
print("Order status updated to shipped")

# Delete
container_client.delete_item(item=order["id"], partition_key="cust-001")
print("Order deleted")

In [ ]:
# Read current throughput and replace with autoscale
offer = container_client.read_offer()
print(f"Current throughput: {offer.offer_throughput} RU/s")

# Switch to autoscale (max 4000 RU/s — scales down to 400 automatically)
container_client.replace_throughput(
    properties={"resource": {"autoscaleSettings": {"maxThroughput": 4000}}}
)
print("Switched to autoscale: 400–4000 RU/s")

# Change feed — read recent changes
items = container_client.query_items_change_feed(is_start_from_beginning=False)
for item in items:
    print(f"Change feed item: {item.get('id')}  status: {item.get('status')}")

## Summary

| Concept | Key Takeaway |
|---|---|
| Azure SQL Database | Fully managed PaaS SQL Server; single DB, elastic pool, or serverless |
| General Purpose tier | Balanced — remote SSD; most workloads |
| Business Critical tier | Local SSD; built-in AG; readable secondary; fast failover |
| Hyperscale tier | Up to 100 TiB; scale-out read replicas; distributed storage layer |
| Serverless tier | Auto-pause/resume; pay per use; ~1 min resume latency |
| SQL Managed Instance | ~100% SQL Server compat; VNet-injected; SQL Agent, cross-DB queries, CLR |
| PITR | Point-in-time restore within 1–35 day retention window |
| LTR | Long-term retention up to 10 years in Blob Storage |
| TDE | Transparent Data Encryption — always on; optional BYOK via Key Vault |
| Always Encrypted | Column encryption — key stays with client; Azure never sees plaintext |
| Cosmos DB | Globally distributed NoSQL; multi-model API; single-digit ms latency; 99.999% SLA |
| Request Units (RU/s) | Universal throughput currency — 1 RU = read 1 KB by ID |
| Partition key | Most critical design decision — high cardinality, even distribution |
| Autoscale throughput | Scales 10%–100% of max RU/s; pay for peak reached |
| Consistency levels | Strong → Bounded Staleness → Session → Consistent Prefix → Eventual |
| Session consistency | Default — read-your-own-writes within a session |
| Multi-region writes | Active-active; conflict resolution via LWW or custom merge |
| Change Feed | Ordered log of inserts+updates; powers event-driven architectures |
| TTL | Auto-expire items after N seconds — no manual cleanup needed |
| Synapse Link | Analytical store — columnar copy for analytics; zero ETL; no OLTP impact |